# Pre- and Post-Test Pedagogical Knowledge Scores for Azerbaijani Regional Teacher Cohorts Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 ("Pre- and Post-Test Pedagogical Knowledge Scores for Azerbaijani Regional Teacher Cohorts") dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is provided via a Croissant schema URL and includes matched cohort records with regional identifiers and pre/post test scores.

In [ ]:
# Install mlcroissant, if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vn9f-6y79/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets and their fields/columns. All elements are referenced by their `@id` for reproducibility.

Here, we enumerate all record sets (`cr:RecordSet`) in the dataset using their `@id`, and for each, list their available fields by `@id` and `name`.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.metadata.record_sets
if isinstance(record_sets, list) and record_sets:
    for rset in record_sets:
        print(f"RecordSet @id: {rset['@id']}")
        if 'field' in rset and rset['field']:
            for field in rset['field']:
                # field could be a dict or an @id ref
                if isinstance(field, dict):
                    print(f"    Field @id: {field['@id']}\tname: {field.get('name', '')}")
                else:
                    print(f"    Field @id: {field}")
        else:
            print("    No fields defined.")
else:
    print("No record sets found in the schema (ensure the dataset is up to date).")

## 3. Data Extraction
Load data from the primary record set into a pandas DataFrame for exploration.

> **Tip:** Use the record set and field `@id`s from the previous overview step for all data references.

In [ ]:
# Retrieve all record set IDs
record_sets = dataset.metadata.record_sets
record_set_ids = []
if isinstance(record_sets, list) and record_sets:
    for rset in record_sets:
        record_set_ids.append(rset['@id'])

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview: show available columns for the first record set
if dataframes:
    primary_record_set_id = record_set_ids[0]
    print(f"Columns in primary record set ({primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()
else:
    print("No dataframes were successfully loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing procedures such as filtering, normalization, and grouping using field `@id`s only.

In [ ]:
import numpy as np
# Example: Let's assume the numeric field is `post_test_mean_score` and grouping by `region`
# Use the actual field `@id`s present in your data; for demonstration, we'll use placeholder ids
# E.g.:
# numeric_field_id = 'cr:post_test_mean_score'
# group_field_id = 'cr:region'

primary_record_set_id = record_set_ids[0] if record_set_ids else None

# Show first 5 rows to help user pick a numeric field
if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    print("First 5 rows of the main table:\n", df.head())
    print("Available columns:", list(df.columns))
    
    # If 'cr:post_test_mean_score' field is present, proceed:
    # Replace with the actual @id for the 'post test mean score' field according to the schema
    # Example field names (these should match your data):
    for col in df.columns:
        if 'post' in col and 'score' in col.lower():
            numeric_field = col
            break
        else:
            numeric_field = df.columns[-1]  # fallback
    print(f"Selected numeric field for analysis: {numeric_field}\n")

    # Set an arbitrary threshold for demonstration; in practice, inspect distribution first
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a field such as 'cr:region' or the first string/categorical column
    group_field = None
    for col in df.columns:
        # Attempt to detect a grouping field
        if 'region' in col.lower() or df[col].dtype == object:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nMean statistics grouped by {group_field} (by @id):")
        print(grouped_df.head())
else:
    print("No primary record set data available for EDA.")

## 5. Visualization
Visualize numeric data distributions and relationships for further insights, using columns referenced by their proper `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Plot histogram for the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=10, color="skyblue")
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field is available, plot grouped boxplots
    if group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for plotting.")

## 6. Conclusion
This notebook demonstrated how to programmatically access, filter, and visualize the FAIR^2 pedagogical knowledge score dataset using the `mlcroissant` library, referencing all entities by their `@id`. Such work can enable reproducible, standards-based data science workflows and facilitate reuse and sharing of educational outcome research.